# Train YOLO26n detection — Strawberry Vision Pi (Phase 2)

Trains the detection model on Zenodo strawberry imagery (record 6126677, 813 images, 3 classes: ripe / unripe / peduncle). Output: `best.pt` weights saved to Drive for download to the Pi.

**Runtime**: switch to `Runtime → Change runtime type → T4 GPU` (free) before running. A100/L4 are faster but T4 finishes Phase 2 detection in ~20–40 min wall-clock.

**Out of scope here**: NCNN export (do that on Mac after pulling `best.pt`); Hailo HEF conversion (Phase 5).

## 1. Environment

In [ ]:
!pip install -q ultralytics
import torch, ultralytics
print(f'ultralytics {ultralytics.__version__}')
print(f'torch       {torch.__version__}')
print(f'CUDA        {torch.cuda.is_available()} / {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none"}')

## 2. Clone the repo

Public read-only HTTPS clone — no auth required.

In [ ]:
%cd /content
!rm -rf strawb-analysis
!git clone https://github.com/AKarode/strawb-analysis.git
%cd strawb-analysis

## 3. Download Zenodo dataset

Pulls record 6126677 (~1.5 GB), extracts the inner `strawberries.zip`, and places it at `data/zenodo/strawberries/{training,validation}/` — the layout `train_detect.py` expects.

In [ ]:
!mkdir -p data/zenodo/_dl
!curl -sSL -o data/zenodo/_dl/zenodo.zip 'https://zenodo.org/api/records/6126677/files-archive'
!unzip -q -o data/zenodo/_dl/zenodo.zip -d data/zenodo/_dl/
!ls data/zenodo/_dl/

In [ ]:
# The outer archive contains a nested strawberries.zip. Unpack it into
# data/zenodo/strawberries/. Defensive: handle both the nested-zip case and
# the case where the outer archive already contained the directories.
import shutil, subprocess, zipfile
from pathlib import Path

dl = Path('data/zenodo/_dl')
target = Path('data/zenodo/strawberries')
target.mkdir(parents=True, exist_ok=True)

nested = next(dl.glob('*.zip'), None)
candidates = [p for p in dl.glob('*.zip') if p.name != 'zenodo.zip']
if candidates:
    inner = candidates[0]
    print(f'extracting nested {inner.name}')
    with zipfile.ZipFile(inner) as zf:
        zf.extractall(dl)

for src_name in ('training', 'validation', 'strawberries.yaml', 'names.txt', 'data.yaml'):
    # Source may be either at dl/<name> or dl/strawberries/<name>
    for src in (dl / src_name, dl / 'strawberries' / src_name):
        if src.exists():
            dst = target / src_name
            if dst.exists():
                if dst.is_dir():
                    shutil.rmtree(dst)
                else:
                    dst.unlink()
            shutil.move(str(src), str(dst))
            break

shutil.rmtree(dl, ignore_errors=True)
print('---')
subprocess.run(['ls', '-la', str(target)])

In [ ]:
# Sanity-check counts. Expected: 654 train images, 159 val images.
!echo 'training jpgs:' $(find data/zenodo/strawberries/training -iname '*.jpg' | wc -l)
!echo 'training txts:' $(find data/zenodo/strawberries/training -iname '*.txt' | wc -l)
!echo 'val      jpgs:' $(find data/zenodo/strawberries/validation -iname '*.jpg' | wc -l)
!echo 'val      txts:' $(find data/zenodo/strawberries/validation -iname '*.txt' | wc -l)
!cat data/zenodo/strawberries/strawberries.yaml 2>/dev/null || cat data/zenodo/strawberries/data.yaml 2>/dev/null || echo 'no upstream yaml'

## 4. Mount Google Drive for checkpoint persistence

Colab runtimes disconnect after ~12 h or on idle. Writing run outputs to Drive means a disconnect doesn't lose epochs.

Skip this cell if you'd rather not mount Drive — just change `--project` below to `runs/detect` and download `best.pt` manually before disconnect.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import pathlib
PROJECT_DIR = '/content/drive/MyDrive/strawb-models/runs/detect'
pathlib.Path(PROJECT_DIR).mkdir(parents=True, exist_ok=True)
print(f'training output dir: {PROJECT_DIR}')

## 5. Train

Defaults: 100 epochs, 640×640, batch 32 (good for T4 16 GB). Tune `--batch` down to 16 on smaller GPUs or up to 64 on A100/L4. Early stopping at `--patience 20` will end the run earlier than 100 epochs once val mAP plateaus.

In [ ]:
!python scripts/train_detect.py \
    --data-root data/zenodo/strawberries \
    --weights yolo26n.pt \
    --epochs 100 \
    --imgsz 640 \
    --batch 32 \
    --device 0 \
    --project "$PROJECT_DIR" \
    --name zenodo_yolo26n \
    --patience 20

## 6. Inspect results

In [ ]:
import os
from IPython.display import Image, display

run_dir = f'{PROJECT_DIR}/zenodo_yolo26n'
for fname in ('results.png', 'confusion_matrix.png', 'confusion_matrix_normalized.png',
              'val_batch0_pred.jpg', 'PR_curve.png'):
    path = os.path.join(run_dir, fname)
    if os.path.exists(path):
        print(f'--- {fname} ---')
        display(Image(path))
    else:
        print(f'(missing) {path}')

In [ ]:
# Per-epoch metrics
import pandas as pd
df = pd.read_csv(f'{PROJECT_DIR}/zenodo_yolo26n/results.csv')
df.tail(10)

## 7. Download `best.pt`

The weights file is already on Drive at `MyDrive/strawb-models/runs/detect/zenodo_yolo26n/weights/best.pt`. Sync it down to your Mac via Drive desktop client, or download directly from the cell below.

In [ ]:
from google.colab import files
best = f'{PROJECT_DIR}/zenodo_yolo26n/weights/best.pt'
files.download(best)

## Next steps (off-Colab)

1. Mac side: `mkdir -p models/detect && cp ~/Downloads/best.pt models/detect/yolo26n_zenodo.pt`
2. Mac side: `python scripts/smoke_test_yolo26.py` (or a dedicated `scripts/export_detect_ncnn.py` once authored) to produce `models/detect/yolo26n_zenodo_ncnn_model/`.
3. Pi side: pull the NCNN model down (rsync from Mac, or commit/fetch via release artifact) and run the CPU benchmark — Phase 4 territory.